# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
pip install -q schedule pytest

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: C:\Users\1\python-hw\Scripts\python.exe -m pip install --upgrade pip


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests
import schedule
from bs4 import BeautifulSoup

## Задание 1. Сбор данных об одной книге (2 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
RATING_VALUES = {
    "One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5,
}

def get_book_data(book_url: str) -> dict:
    """Скачать страницу книги и вернуть сведения о ней.

    Args:
        book_url (str): Ссылка на страницу книги.

    Returns:
        dict: Название, рейтинг, количество в наличии, описание,
        UPC, тип продукта, цены без налога и с налогом, налог
        и количество отзывов.

    Raises:
        requests.RequestException: При сетевой ошибке или HTTP-ошибке.
        ValueError: Если обязательные элементы страницы отсутствуют.
    """
    response = requests.get(book_url, timeout=10)
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    main = soup.select_one(".product_main")
    if main is None:
        raise ValueError(f"Не найдена карточка книги: {book_url}")

    title = main.find("h1")
    rating = main.select_one(".star-rating")
    stock = main.select_one(".availability")
    if any(node is None for node in (title, rating, stock)):
        raise ValueError(f"Не найдены обязательные поля: {book_url}")

    score = None
    for name in rating.get("class", []):
        if name in RATING_VALUES:
            score = RATING_VALUES[name]
            break

    stock_match = re.search(r"\d+", stock.get_text())

    description_heading = soup.select_one("#product_description")
    description = (
        description_heading.find_next_sibling("p")
        if description_heading is not None else None
    )

    book_data = {
        "title": title.get_text(strip=True),
        "rating": score,
        "availability": int(stock_match.group()) if stock_match else 0,
        "description": (
            description.get_text(" ", strip=True)
            if description is not None else ""
        ),
    }

    for row in soup.select("table.table-striped tr"):
        key, value = row.find("th"), row.find("td")
        if key is not None and value is not None:
            field = key.get_text(strip=True)
            if field != "Availability":
                book_data[field] = value.get_text(strip=True)

    return book_data

In [4]:
# Используйте для самопроверки
book_url = 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'rating': 3,
 'availability': 22,
 'description': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you up there,And your c

## Задание 2. Сбор данных обо всех книгах (2 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
def scrape_books(
    is_save: bool = False,
    n: int | None = None,
) -> list[dict]:
    """Собрать данные о книгах каталога.

    Args:
        is_save (bool): Сохранить результат в books_data.txt.
        n (int | None): Максимальное число страниц.
            Если None, собрать все страницы.

    Returns:
        list[dict]: Данные о собранных книгах.

    Raises:
        ValueError: Если n не является положительным целым числом.
    """
    if n is not None and (type(n) is not int or n < 1):
        raise ValueError("n должно быть положительным целым числом")

    books = []
    page = 1

    while n is None or page <= n:
        page_url = (
            f"https://books.toscrape.com/catalogue/page-{page}.html"
        )
        response = requests.get(page_url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        for link in soup.select("article.product_pod h3 a"):
            book_url = urljoin(page_url, link["href"])
            books.append(get_book_data(book_url))
            time.sleep(0.2)

        if soup.select_one("li.next a") is None:
            break

        page += 1

    if is_save:
        project_dir = Path.cwd()

        if project_dir.name == "notebooks":
            project_dir = project_dir.parent
            
        output_path = project_dir / "artifacts" / "books_data.txt"
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with output_path.open("w", encoding="utf-8") as file:
            json.dump(books, file, ensure_ascii=False, indent=2)

        print(f"Файл сохранён: {output_path}")

    return books

In [6]:
# Проверка работоспособности функции
res = scrape_books(is_save=True, n=1) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Файл сохранён: C:\Users\1\Documents\books_scraper\artifacts\books_data.txt
<class 'list'> 20


## Задание 3. Настройка регулярной выгрузки (1 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [7]:
def scheduled_scrape() -> None:
    """Собрать и сохранить книги; сообщить об ошибке, не остановив цикл."""
    try:
        books = scrape_books(is_save=True)
        print(f"Сохранено книг: {len(books)}")
    except (requests.RequestException, ValueError, OSError) as error:
        print(f"Выгрузка не выполнена: {error}")
        
def run_scheduler(at_time: str = "19:00") -> None:
    """Запустить ежедневную выгрузку в указанное местное время.
    
     Args:
        at_time (str): Время запуска в локальном часовом поясе компьютера.
    """
    scheduler = schedule.Scheduler()
    scheduler.every().day.at(at_time).do(scheduled_scrape)

    print(f"Ежедневная выгрузка в {at_time}")

    try:
        while True:
            scheduler.run_pending()
            time.sleep(1)
    except KeyboardInterrupt:
        print("Планировщик остановлен")

run_scheduler("19:00")

Ежедневная выгрузка в 19:00
Планировщик остановлен


## Задание 4. Написание автотестов (1,5 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [8]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

! pytest tests/test_scraper.py

============================= test session starts =============================
platform win32 -- Python 3.11.11, pytest-7.4.3, pluggy-1.5.0
rootdir: C:\Users\1\Documents\books_scraper
plugins: anyio-4.6.2
collected 3 items

tests\test_scraper.py ...                                                [100%]

============================== 3 passed in 0.52s ==============================


## Задание 5. Оформление проекта на GitHub и работа с Git (3,5 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

## Выводы, рефлексия и сдача

### Рефлексия после выполнения (заполните здесь)

**1. Что было задано и что получилось**:

`Ответ: Было задано сделать парсер книжного магазина с регулярной выгрузкой. Помимо этого были заданы автотесты и введена система контроля версий.`

**2. Как выполняли**: что сработало сразу, где застряли и как выбрались:

`Ответ: Я очень мало сталикивался с git, но знаком с концепцией и особенностями, и все шаги перепроверял со встроенным в гугл ИИ. Еще застрял с тестами, подробнее в п.6.`

**3. Использование ИИ**: цель, использованные промпты, где помощник ошибся и как вы это поняли. Если не использовали, так и напишите:

`Ответ: Описал выше в п.2. Также попросил собрать gitignore и приукрасить README`

**4. Что осталось непонятным** (честный список, он поможет на консультации):

`Ответ: В целом все ясно, поскольку задачи небольшие`

**5. Самооценка по критериям задания**:

`Ответ: Зачет`

**6. Дополнительный вопрос: обратная связь по заданию.** Что было ценно, что сложно, а что, на ваш взгляд, требует доработки?

`Ответ: Очень смутил момент с тестированием, понял лишь когда еще раз перечитал все задание до конца. Указание с созданием scraper.py появляется лишь в самом конце, когда как тесты мы пишем непосредственно для scraper.py. Я даже по началу начал изучать возможность проводить тесты ipynb с сохранением ячейки с запуском через '!', однако вовремя решил еще раз изучить задание до конца. Мне кажется, лучше явно указать где-то в начале или в блоке с заданием по тестированию, что весь код должен быть продублирован в отдельный модуль scraper.py.`

> Напоминание: рефлексия пишется своими словами, без помощи ИИ.

